# Floorplan YOLOv8-seg training (Colab)

**Runtime → Change runtime type → GPU (T4 is enough).**

1. Upload `artifacts/dataset.zip` to Google Drive as `MyDrive/image_drawing_models/dataset.zip`
   (or set `ZIP_PATH` below).
2. Runtime → Run all.
3. Download `MyDrive/image_drawing_models/best.pt` and run locally:
   `./scripts/deploy_model.sh ~/Downloads/best.pt`

**Time / cost:** T4 free tier ≈ 30–90 min for 40 epochs (yolov8n-seg, batch 16). A100 faster; not required. Colab Pro / RunPod only if free GPU quota is gone.

In [ ]:
# Optional: mount Drive (skip if you upload dataset.zip via Colab Files UI)
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = '/content/drive/MyDrive/image_drawing_models/dataset.zip'
OUT_DIR = '/content/drive/MyDrive/image_drawing_models'
DATA_ROOT = '/content/datasets/floorplans_dataset'

In [ ]:
import os, zipfile, shutil
from pathlib import Path

!pip -q install ultralytics opencv-python-headless

assert Path(ZIP_PATH).is_file(), f'Missing {ZIP_PATH}. Upload dataset.zip to Drive first.'
if Path(DATA_ROOT).exists():
    shutil.rmtree(DATA_ROOT)
Path('/content/datasets').mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ZIP_PATH) as z:
    z.extractall('/content/datasets')
# zip may contain datasets/floorplans_dataset/... or floorplans_dataset/...
cands = list(Path('/content/datasets').rglob('data.yaml'))
assert cands, 'data.yaml not found after unzip'
data_yaml = cands[0]
root = data_yaml.parent
print('data_yaml', data_yaml)
print('train images', len(list((root / 'images' / 'train').glob('*'))))
print('val images', len(list((root / 'images' / 'val').glob('*'))))
# Rewrite path for Colab
data_yaml.write_text(
    f"path: {root.resolve()}\n"
    "train: images/train\n"
    "val: images/val\n"
    "nc: 5\n"
    "names: ['wall', 'door', 'window', 'stair', 'partition']\n"
)

In [ ]:
import torch
from ultralytics import YOLO
from pathlib import Path

print('cuda', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
assert torch.cuda.is_available(), 'Enable GPU: Runtime → Change runtime type → T4'

data_yaml = list(Path('/content/datasets').rglob('data.yaml'))[0]
model = YOLO('yolov8n-seg.pt')  # or upload a previous floorplan-seg.pt and pass its path

results = model.train(
    data=str(data_yaml),
    epochs=40,
    imgsz=640,
    batch=16,
    device=0,
    patience=12,
    workers=2,
    project='/content/runs',
    name='floorplan_seg',
    exist_ok=True,
    save_period=5,
    plots=True,
)

best = Path(results.save_dir) / 'weights' / 'best.pt'
print('best', best, 'size', best.stat().st_size)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)
dest = Path(OUT_DIR) / 'best.pt'
shutil.copy2(best, dest)
print('Copied to', dest)

### Optional: load zip via public Drive link + gdown
```python
!pip -q install gdown
!gdown --fuzzy 'YOUR_SHARE_LINK' -O /content/dataset.zip
ZIP_PATH = '/content/dataset.zip'
```